# 📚 elib-recommender — Book Recommendation Engine

> **ML Assignment Project** | Built to power the E-Library app

### Objective
Build a **collaborative filtering** book recommendation system trained on the **Goodreads 10k public dataset**. The model learns from millions of real user ratings to predict which books a user is most likely to enjoy.

### Approach
- **Matrix Factorization** using `TruncatedSVD` from scikit-learn
- **User-Item Matrix** built from Goodreads ratings data
- **Evaluation** using RMSE and cosine similarity

### Pipeline
1. Install & Import Libraries
2. Download Dataset
3. Exploratory Data Analysis (EDA)
4. Data Preprocessing
5. Model Training (SVD Matrix Factorization)
6. Model Evaluation
7. Export Model
8. Live Demo — Get Recommendations

---
## 📦 Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import joblib
import requests
import warnings
import os

from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.dpi'] = 120
sns.set_style('darkgrid')

print('✅ All libraries imported successfully!')
print(f'   NumPy   : {np.__version__}')
print(f'   Pandas  : {pd.__version__}')

---
## 📥 Step 2 — Download the Goodreads Dataset

We use the **Goodreads Books 10k dataset** from GitHub:
- `ratings.csv` — ~6 million user-book ratings (scale 1–5)
- `books.csv` — metadata for 10,000 books (title, author, cover, avg rating)

In [ ]:
BASE_URL = 'https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/master/'

print('⬇️  Downloading ratings dataset...')
ratings_df = pd.read_csv(BASE_URL + 'ratings.csv')

print('⬇️  Downloading books metadata...')
books_df = pd.read_csv(BASE_URL + 'books.csv', low_memory=False)

print(f'\n✅ Ratings : {ratings_df.shape[0]:,} rows × {ratings_df.shape[1]} columns')
print(f'✅ Books   : {books_df.shape[0]:,} rows × {books_df.shape[1]} columns')

---
## 🔍 Step 3 — Exploratory Data Analysis (EDA)

In [ ]:
print('=== Ratings Sample ===')
display(ratings_df.head())

print('\n=== Books Sample ===')
display(books_df[['book_id', 'title', 'authors', 'average_rating', 'ratings_count']].head())

print(f'\n📊 Dataset Stats:')
print(f'   Unique users : {ratings_df["user_id"].nunique():,}')
print(f'   Unique books : {ratings_df["book_id"].nunique():,}')
print(f'   Rating range : {ratings_df["rating"].min()} – {ratings_df["rating"].max()}')
print(f'   Avg rating   : {ratings_df["rating"].mean():.2f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Goodreads Dataset — Exploratory Analysis', fontsize=15, fontweight='bold', y=1.02)

# 1. Rating Distribution
rating_counts = ratings_df['rating'].value_counts().sort_index()
axes[0].bar(rating_counts.index, rating_counts.values, color='#6C63FF', edgecolor='white', width=0.6)
axes[0].set_title('Rating Distribution', fontweight='bold')
axes[0].set_xlabel('Rating (1–5)')
axes[0].set_ylabel('Number of Ratings')
for i, v in zip(rating_counts.index, rating_counts.values):
    axes[0].text(i, v + 20000, f'{v/1e6:.1f}M', ha='center', fontsize=9)

# 2. Ratings Per User
ratings_per_user = ratings_df.groupby('user_id').size()
axes[1].hist(ratings_per_user[ratings_per_user <= 200], bins=40, color='#FF6584', edgecolor='white')
axes[1].set_title('Ratings Per User (≤200)', fontweight='bold')
axes[1].set_xlabel('Number of Books Rated')
axes[1].set_ylabel('Number of Users')
axes[1].axvline(ratings_per_user.median(), color='white', linestyle='--', label=f'Median: {ratings_per_user.median():.0f}')
axes[1].legend()

# 3. Book Average Rating Distribution
axes[2].hist(books_df['average_rating'].dropna(), bins=30, color='#43D9AD', edgecolor='white')
axes[2].set_title('Book Avg Rating Distribution', fontweight='bold')
axes[2].set_xlabel('Average Rating')
axes[2].set_ylabel('Number of Books')
axes[2].axvline(books_df['average_rating'].mean(), color='white', linestyle='--',
                label=f'Mean: {books_df["average_rating"].mean():.2f}')
axes[2].legend()

plt.tight_layout()
plt.savefig('../eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 EDA plot saved as eda_overview.png')

In [ ]:
# Top 10 most rated books
top10 = books_df.nlargest(10, 'ratings_count')[['title', 'authors', 'average_rating', 'ratings_count']]
top10['ratings_count'] = top10['ratings_count'].apply(lambda x: f'{x:,}')

print('📚 Top 10 Most Rated Books on Goodreads:')
display(top10.reset_index(drop=True))

---
## 🧹 Step 4 — Data Preprocessing

We filter out:
- Users who rated fewer than **10 books** (not enough signal)
- Build a **User-Item Matrix** where rows = users, columns = books, values = ratings

In [ ]:
# Filter active users (rated at least 10 books)
user_counts = ratings_df['user_id'].value_counts()
active_users = user_counts[user_counts >= 10].index
ratings_filtered = ratings_df[ratings_df['user_id'].isin(active_users)].copy()

print(f'Original ratings : {len(ratings_df):,}')
print(f'Filtered ratings : {len(ratings_filtered):,}')
print(f'Active users     : {ratings_filtered["user_id"].nunique():,}')
print(f'Books covered    : {ratings_filtered["book_id"].nunique():,}')

# Sample for faster training (use full set if Colab/powerful machine)
SAMPLE_USERS = 5000
sampled_users = np.random.choice(ratings_filtered['user_id'].unique(), SAMPLE_USERS, replace=False)
ratings_sample = ratings_filtered[ratings_filtered['user_id'].isin(sampled_users)]
print(f'\n🎯 Working with {SAMPLE_USERS:,} sampled users for training')
print(f'   Sample ratings: {len(ratings_sample):,}')

In [ ]:
# Build User-Item Matrix
print('Building User-Item Matrix...')
user_item_matrix = ratings_sample.pivot_table(
    index='user_id',
    columns='book_id',
    values='rating',
    fill_value=0
)

print(f'✅ User-Item Matrix shape: {user_item_matrix.shape}')
print(f'   Rows    = {user_item_matrix.shape[0]:,} users')
print(f'   Columns = {user_item_matrix.shape[1]:,} books')
sparsity = 1 - (ratings_sample.shape[0] / (user_item_matrix.shape[0] * user_item_matrix.shape[1]))
print(f'   Matrix Sparsity: {sparsity:.2%} (most entries are 0 — typical for recommendation systems)')

---
## 🤖 Step 5 — Model Training (SVD Matrix Factorization)

**How it works:**
- The User-Item matrix is huge and sparse
- SVD decomposes it into smaller **latent factor** matrices
- These latent factors capture hidden patterns (e.g., "user likes dark fantasy" without explicitly knowing)
- We use these factors to predict unseen ratings

In [ ]:
from sklearn.decomposition import TruncatedSVD

matrix = user_item_matrix.values  # numpy array

# Train SVD model
N_COMPONENTS = 50  # number of latent factors
print(f'🚀 Training TruncatedSVD with {N_COMPONENTS} latent factors...')

svd = TruncatedSVD(n_components=N_COMPONENTS, random_state=42)
user_factors = svd.fit_transform(matrix)           # shape: (users, n_components)
item_factors = svd.components_                     # shape: (n_components, books)

print(f'✅ Training complete!')
print(f'   User factors shape : {user_factors.shape}')
print(f'   Item factors shape : {item_factors.shape}')
print(f'   Variance explained : {svd.explained_variance_ratio_.sum():.2%}')

In [ ]:
# Explained Variance Plot
plt.figure(figsize=(10, 4))
cumvar = np.cumsum(svd.explained_variance_ratio_)
plt.plot(range(1, N_COMPONENTS + 1), cumvar, color='#6C63FF', linewidth=2, marker='o', markersize=4)
plt.fill_between(range(1, N_COMPONENTS + 1), cumvar, alpha=0.2, color='#6C63FF')
plt.axhline(0.8, color='#FF6584', linestyle='--', label='80% threshold')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('SVD — Cumulative Explained Variance by Components', fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../svd_variance.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 📊 Step 6 — Model Evaluation

In [ ]:
# Reconstruct full ratings matrix from SVD factors
reconstructed = np.dot(user_factors, item_factors)  # predicted ratings

# Evaluate only on non-zero (actual) ratings
actual_mask = matrix > 0
actual_vals = matrix[actual_mask]
predicted_vals = reconstructed[actual_mask]

rmse = np.sqrt(mean_squared_error(actual_vals, predicted_vals))
mae  = np.mean(np.abs(actual_vals - predicted_vals))

print('=== Model Evaluation ===')
print(f'RMSE (Root Mean Squared Error) : {rmse:.4f}')
print(f'MAE  (Mean Absolute Error)     : {mae:.4f}')
print(f'\nInterpretation: On average, predicted ratings are off by ~{mae:.2f} stars on a 1–5 scale.')

In [ ]:
# Sample 2000 points for visualization
sample_idx = np.random.choice(len(actual_vals), min(2000, len(actual_vals)), replace=False)

plt.figure(figsize=(8, 6))
plt.scatter(actual_vals[sample_idx], predicted_vals[sample_idx],
            alpha=0.3, color='#6C63FF', s=15, label='Predictions')
plt.plot([1, 5], [1, 5], 'r--', linewidth=2, label='Perfect prediction')
plt.xlabel('Actual Rating')
plt.ylabel('Predicted Rating')
plt.title(f'Actual vs Predicted Ratings\nRMSE={rmse:.4f} | MAE={mae:.4f}', fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 💾 Step 7 — Export Model

In [ ]:
# Save model artifacts
os.makedirs('../models', exist_ok=True)
os.makedirs('../data', exist_ok=True)

# Save SVD model
joblib.dump(svd, '../models/svd_model.pkl')

# Save user-item matrix column order (needed for inference)
joblib.dump(list(user_item_matrix.columns), '../models/book_ids.pkl')
joblib.dump(list(user_item_matrix.index), '../models/user_ids.pkl')

# Save user factors (for fast lookup)
np.save('../models/user_factors.npy', user_factors)
np.save('../models/item_factors.npy', item_factors)

# Save clean book metadata
books_clean = books_df[['book_id','title','authors','average_rating','ratings_count','image_url']].copy()
books_clean.dropna(subset=['title','authors'], inplace=True)
books_clean.to_csv('../data/books_clean.csv', index=False)

print('✅ Exported:')
print('   models/svd_model.pkl')
print('   models/book_ids.pkl')
print('   models/user_ids.pkl')
print('   models/user_factors.npy')
print('   models/item_factors.npy')
print('   data/books_clean.csv')

---
## 🎯 Step 8 — Live Demo: Get Recommendations

In [ ]:
def get_top_n_recommendations(user_index, top_n=10):
    """
    Given a user's row index in the matrix, return top N book recommendations.
    Excludes books the user has already rated.
    """
    # Predict ratings for all books for this user
    user_vec = user_factors[user_index].reshape(1, -1)  # (1, n_components)
    predicted_ratings = np.dot(user_vec, item_factors).flatten()  # (n_books,)

    # Mask already-rated books
    already_rated = matrix[user_index] > 0
    predicted_ratings[already_rated] = -1  # exclude these

    # Get top N book indices
    top_indices = np.argsort(predicted_ratings)[::-1][:top_n]
    book_ids_list = user_item_matrix.columns.tolist()

    results = []
    for idx in top_indices:
        book_id = book_ids_list[idx]
        book_info = books_clean[books_clean['book_id'] == book_id]
        if not book_info.empty:
            results.append({
                'title': book_info.iloc[0]['title'],
                'authors': book_info.iloc[0]['authors'],
                'avg_rating': round(float(book_info.iloc[0]['average_rating']), 2),
                'predicted_score': round(float(predicted_ratings[idx]), 3)
            })
    return results


# ── Demo ─────────────────────────────────────────────────────────────────────
TEST_USER_INDEX = 0  # First user in our sample
actual_user_id = user_item_matrix.index[TEST_USER_INDEX]
n_rated = int((matrix[TEST_USER_INDEX] > 0).sum())

print(f'📖 User ID  : {actual_user_id}')
print(f'   Books rated by this user: {n_rated}')
print(f'\n🎯 Top 10 Recommendations:\n')

recs = get_top_n_recommendations(TEST_USER_INDEX, top_n=10)
for i, rec in enumerate(recs, 1):
    print(f"{i:2}. {rec['title']}")
    print(f"    👤 {rec['authors']}")
    print(f"    ⭐ Avg: {rec['avg_rating']} | Predicted Score: {rec['predicted_score']}")
    print()

In [ ]:
def find_similar_books(book_id, top_n=5):
    """
    Content-based: find books similar to a given book_id
    using cosine similarity between item factor vectors.
    """
    from sklearn.metrics.pairwise import cosine_similarity

    book_ids_list = user_item_matrix.columns.tolist()
    if book_id not in book_ids_list:
        return f'Book ID {book_id} not in matrix'

    idx = book_ids_list.index(book_id)
    book_vec = item_factors[:, idx].reshape(1, -1)  # (1, n_components)
    all_vecs = item_factors.T                        # (n_books, n_components)

    similarities = cosine_similarity(book_vec, all_vecs).flatten()
    similarities[idx] = -1  # exclude itself

    top_indices = np.argsort(similarities)[::-1][:top_n]

    results = []
    for i in top_indices:
        bid = book_ids_list[i]
        info = books_clean[books_clean['book_id'] == bid]
        if not info.empty:
            results.append({
                'title': info.iloc[0]['title'],
                'authors': info.iloc[0]['authors'],
                'similarity': round(float(similarities[i]), 4)
            })
    return results


# Demo: Find books similar to book_id = 1 (The Hunger Games)
TARGET_BOOK_ID = 1
target_info = books_clean[books_clean['book_id'] == TARGET_BOOK_ID].iloc[0]

print(f'📖 Finding books similar to: "{target_info["title"]}" by {target_info["authors"]}\n')
similar = find_similar_books(TARGET_BOOK_ID, top_n=5)
for i, book in enumerate(similar, 1):
    print(f"{i}. {book['title']} — {book['authors']}")
    print(f"   Similarity Score: {book['similarity']}")
    print()

---
## ✅ Summary

| Step | Description | Status |
|------|-------------|--------|
| Dataset | Goodreads 10k — 6M ratings, 10k books | ✅ |
| EDA | Rating distribution, user activity, book popularity | ✅ |
| Preprocessing | Filtered inactive users, built User-Item matrix | ✅ |
| Model | TruncatedSVD (Matrix Factorization) | ✅ |
| Evaluation | RMSE, MAE, Actual vs Predicted plot | ✅ |
| Export | model.pkl + book metadata | ✅ |
| Demo — User | Top-N personalized recommendations | ✅ |
| Demo — Book | Similar books via cosine similarity | ✅ |

### 🔗 Integration with E-Library
The exported model artifacts are loaded by `api/app.py` (Flask).
The E-Library frontend can call:
- `POST /recommend` → personalized recommendations for a user
- `POST /recommend/book` → similar books for a given title